# Directly-Trained Synthetic Eval Results

This notebook plots aggregate per-example accuracy against the number of assignments in the evaluation metadata. Rows are the model's training dataset (r-trivial, aperiodic, periodic, then the two sensitivity-ordered periodic conditions) and columns are the evaluation dataset (r-trivial, aperiodic, periodic).

Each line is one 275M model variant: hue encodes the model family (transformer=red, hybrid=purple, GDN=blue) and the marker the within-family variant (circles for the canonical RoPE/full variants, squares for the restricted NoPE/SDP ones). The encoding lives in `aesthetics.py` so other notebooks share it. The whole analysis keeps only in-distribution padding (filler proportion <= 25% with easy distractors, matching the training filler config): the directly-trained models are not being tested for padding generalization, so the heavier eval-only padding conditions are excluded everywhere. Evaluation datasets and assignment counts are joined from the source HF dataset by each prediction's `native_id`, since jobs launched with `--no-save-requests` don't store per-example request metadata.

In [ ]:
from __future__ import annotations

from collections import defaultdict
from functools import lru_cache
import json
import math
import re
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportion_confint

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "aesthetics.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

import aesthetics as aes  # noqa: E402
import results_io as rio  # noqa: E402

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 140)

## Config

In [ ]:
RESULTS_ROOT = PROJECT_ROOT / "results"
# Jobs are selected rather than grouped: every group directory under
# results/ is scanned and the newest job wins per model/task, so a new sweep
# is picked up without naming its group here. These three knobs narrow the
# scan (all optional, and applied before any prediction file is opened):
#   RESULT_GROUPS       -- only these group directories, None for all
#   RESULTS_SINCE       -- drop jobs older than this timestamp
#   RESULT_EXPERIMENT_IDS -- pin the analysis to specific jobs
# The date cutoff matters here: runs before August 2026 used the coarse
# tokens_0_100k strata, whose tasks are distinct names that would otherwise
# be pooled alongside the current per-tier ones.
RESULT_GROUPS = ["*-dt*"]
RESULTS_SINCE = "2026-08-25"
RESULT_EXPERIMENT_IDS = None

# Pool assignment counts into powers-of-two bins, matching the long-context plots.
MIN_BIN_EXAMPLES = 3
# Fallback only. The plots derive chance from each subset's candidate
# counts, which are two for two-variable samples and five otherwise.
CHANCE_ACCURACY = 0.20
CHANCE_LINE_COLOR = aes.CHANCE_LINE_COLOR
ASSIGNMENT_REFERENCE = 256
ASSIGNMENT_REFERENCE_COLOR = "#2CA02C"
# The dataset's extra-assignment sweep starts at 8 (lowered from 32 in the
# floor-8 regeneration); pin the x-axis floor there instead of deriving it from
# whichever jobs happen to be fetched.
ASSIGNMENT_AXIS_MIN = 10
# Directly-trained models saw filler proportions {0, 0.25} with easy distractors
# (configs/experiments/default.yaml in state-bench); the eval set additionally
# carries heavier eval-only padding (up to 90%, medium/hard distractors). This
# analysis is not about padding generalization, so every aggregation keeps only
# in-distribution padding. Without the filter, the heavy padding-grid rows --
# which exist only at depth 32 -- render as off-trend dips with tight CIs.
TRAIN_MAX_FILLER_PROPORTION = 0.25
TRAIN_DISTRACTION_LEVELS = frozenset({None, "easy"})

# Columns of the train x eval grids: the three evaluation complexity classes.
EVAL_DATASET_ORDER = ["r-trivial", "aperiodic", "periodic"]
# Rows: the five training distributions (the sensitivity-ordered conditions
# train on a repartition of the periodic data).
# The shuffled-order periodic evals run last; flip this on once they are
# fetched to add the Periodic (shuffled) rows/panels back into every figure.
INCLUDE_SENS_SHUFFLED = False

TRAIN_DATASET_ORDER = [
    *EVAL_DATASET_ORDER,
    *(["periodic-sens-shuffled"] if INCLUDE_SENS_SHUFFLED else []),
    "periodic-sens-curriculum",
]
DATASET_LABEL = {
    "r-trivial": "R-trivial",
    "aperiodic": "Aperiodic",
    "periodic": "Periodic",
    "periodic-sens-shuffled": "Periodic (shuffled)",
    "periodic-sens-curriculum": "Periodic (curriculum)",
}


# The single-row summary figures come in two groups: the three original
# training conditions, and the periodic-family contrast (plain periodic vs the
# sensitivity-ordered conditions, all evaluated on periodic).
ORIGINAL_TRAIN_DATASETS = list(EVAL_DATASET_ORDER)
PERIODIC_TRAIN_DATASETS = [d for d in TRAIN_DATASET_ORDER if d.startswith("periodic")]


def matched_eval_dataset(training_dataset: str) -> str:
    """The evaluation dataset distribution-matched to one training dataset."""
    return "periodic" if training_dataset.startswith("periodic-sens") else training_dataset


# Hue encodes the model family (transformer=red, hybrid=purple, GDN=blue) at the
# 275M shade, and markers distinguish the within-family variants (circles for
# RoPE/full, squares for NoPE/SDP). The encoding lives in aesthetics.py so
# pt_results can share it.
SERIES_ORDER = list(aes.STATE_BENCH_MODEL_ORDER)
SERIES_PALETTE = {series: aes.state_bench_model_color(series) for series in SERIES_ORDER}
SERIES_MARKERS = dict(aes.STATE_BENCH_MODEL_MARKERS)
SERIES_LABELS = dict(aes.STATE_BENCH_MODEL_LABELS)

SAVE_FIGURE = True
FIGURE_PATH = PROJECT_ROOT / "state-bench-paper" / "figures" / "dt_accuracy_by_assignments.pdf"

## Load Per-Example Accuracy and Evaluation Metadata

In [ ]:
TRAINING_METADATA_RE = re.compile(
    r"(?P<architecture>transformer-rope|transformer-nope|hybrid|gdn-sdp|gdn-full)[-_/]"
    r"integer-code--(?P<training_dataset>r-trivial|aperiodic"
    r"|periodic--sens-shuffled|periodic--sens-curriculum|periodic)(?:[_/]|$)",
    flags=re.IGNORECASE,
)
TRAINING_SEED_RE = re.compile(
    r"(?:^|[_-])init_seed(?P<seed>\d+)(?:[_-]|$)",
    flags=re.IGNORECASE,
)
TRAINING_SEED_COMPONENT_RE = re.compile(
    r"[_-]init_seed\d+(?=[_-]|$)",
    flags=re.IGNORECASE,
)
EXPERIMENT_NAME_SUFFIX_RE = re.compile(r"-[0-9a-f]{4}$", flags=re.IGNORECASE)

DOC_METADATA_FIELDS = (
    "complexity",
    "num_assignments",
    "token_length",
    "max_dependency_length",
    "num_filler_lines",
    "filler_proportion",
    "filler_distraction",
    "sensitivity_coverage",
)
DOC_METADATA_CACHE_DIR = RESULTS_ROOT / "doc_metadata"


def training_metadata_for(result: dict) -> dict:
    haystack = f"{result.get('experiment_name', '')} {result.get('model', '')}"
    match = TRAINING_METADATA_RE.search(haystack)
    if match is None:
        raise ValueError(f"Could not parse training metadata from: {haystack}")
    seed_match = TRAINING_SEED_RE.search(haystack)
    return {
        "architecture": match["architecture"].lower(),
        # Directory names double the dash before "sens"; series keys use one.
        "training_dataset": match["training_dataset"].lower().replace("--sens-", "-sens-"),
        "training_seed": int(seed_match["seed"]) if seed_match is not None else 0,
    }


@lru_cache(maxsize=None)
def doc_metadata_for_source(
    path: str, subset: str, split: str, revision: str | None = None
) -> dict[str, dict]:
    """Map example id -> doc metadata for one dataset split, cached on disk.

    Jobs launched with --no-save-requests don't store per-example request
    metadata, so predictions are joined back to the source dataset by
    native_id. rio.cached_doc_metadata keys the on-disk cache by the resolved
    commit SHA, so a regenerated dataset never reuses stale metadata.
    """
    return rio.cached_doc_metadata(
        path,
        subset,
        split,
        DOC_METADATA_FIELDS,
        DOC_METADATA_CACHE_DIR,
        revision=revision,
    )


def doc_metadata_for_task(task: dict) -> dict[str, dict]:
    """Return the id -> doc metadata map for a task's HF data source."""
    source = task.get("config", {}).get("data_source") or {}
    if source.get("source_type") != "hf":
        raise ValueError(f"Unsupported data source for {task.get('task')}: {source}")
    return doc_metadata_for_source(
        source["path"], source["subset"], source["split"], source.get("revision")
    )


def evaluation_metadata_for(doc: dict) -> dict:
    """Read the evaluation dataset and assignment count from doc metadata."""
    dataset = str(doc.get("complexity", "")).lower()
    if dataset not in DATASET_LABEL:
        raise ValueError(f"Unexpected complexity in doc metadata: {dataset!r}")
    return {
        "evaluation_dataset": dataset,
        "assignment_count": int(doc["num_assignments"]),
    }


def is_in_distribution_padding(doc: dict) -> bool:
    """Whether an eval example's padding matches the training filler config."""
    return (
        float(doc.get("filler_proportion") or 0.0) <= TRAIN_MAX_FILLER_PROPORTION
        and doc.get("filler_distraction") in TRAIN_DISTRACTION_LEVELS
    )


def experiment_family_name(experiment_name: str) -> str:
    """Normalize seed replicas to their shared experiment configuration."""
    without_seed = TRAINING_SEED_COMPONENT_RE.sub("", experiment_name)
    return EXPERIMENT_NAME_SUFFIX_RE.sub("", without_seed)


def artifact_file_for_task(result: dict, task: dict, kind: str) -> Path | None:
    """Locate a task artifact inside the results group its job came from."""
    return rio.artifact_file(result, task, kind)


def is_integer_code_task(task: dict) -> bool:
    """Return whether an evaluation task uses the Integer Code formatting."""
    source = task.get("config", {}).get("data_source") or {}
    subset = str(source.get("subset", ""))
    return subset.startswith("integer-code--") or str(task.get("task", "")).startswith(
        "state_bench_integer_code_"
    )


def metric_value(instance_metrics: dict, metric: str | None) -> float | None:
    value = instance_metrics
    for key in (metric or "").split(":"):
        if not isinstance(value, dict) or key not in value:
            return None
        value = value[key]
    return float(value) if isinstance(value, int | float) else None


def chance_accuracy(prediction: dict) -> float:
    """Chance accuracy for one instance: one over its number of candidates.

    StateBench varies how many candidate final states it offers -- two for a
    two-variable sample, five otherwise -- so chance is not the flat 1/5 that a
    fixed candidate count would imply.
    """
    candidates = prediction.get("model_output") or ()
    if not candidates:
        raise ValueError(f"Prediction {prediction.get('doc_id')} scored no candidates")
    return 1.0 / len(candidates)


def weighted_chance(data: pd.DataFrame | None) -> float:
    """Example-weighted chance accuracy over a plotted subset."""
    if data is None or data.empty:
        return CHANCE_ACCURACY
    examples = data["examples"].sum()
    return data["chance_total"].sum() / examples if examples else CHANCE_ACCURACY


def add_chance_line(ax: plt.Axes, data: pd.DataFrame | None = None) -> None:
    """Mark chance for the samples in `data`, falling back to the nominal rate."""
    ax.axhline(
        weighted_chance(data), color=CHANCE_LINE_COLOR, linestyle="--", linewidth=1.25
    )


def add_assignment_reference_line(ax: plt.Axes) -> None:
    ax.axvline(
        ASSIGNMENT_REFERENCE,
        color=ASSIGNMENT_REFERENCE_COLOR,
        linestyle=":",
        linewidth=1.6,
    )

In [ ]:
# Selection runs on experiment metadata alone, so superseded predictions are
# never read off disk: load_experiments applies the group/date/id filters and
# select_latest keeps the newest job per model/task, both before the loops
# below open a single artifact.
loaded_results = rio.load_experiments(
    RESULTS_ROOT,
    groups=RESULT_GROUPS,
    since=RESULTS_SINCE,
    experiment_ids=RESULT_EXPERIMENT_IDS,
)
raw_results = rio.select_latest(loaded_results)
superseded_results = rio.superseded_table(loaded_results, raw_results)
print(
    f"Selected {len(raw_results)} of {len(loaded_results)} job(s), "
    f"{sum(len(result['tasks']) for result in raw_results):,} task results"
)
display(rio.selection_table(raw_results))
if not superseded_results.empty:
    print("Superseded by newer jobs:")
    display(superseded_results)


def task_matches_source(result: dict, task: dict) -> bool:
    """Whether a task's predictions came from the dataset revision in use now.

    Regenerating the source dataset reissues every example id, so predictions
    scored against an earlier revision can no longer be joined to it. Checking
    one prediction is enough to tell the generations apart, and doing it here
    keeps mismatched jobs out of the aggregation loops below.
    """
    prediction_path = artifact_file_for_task(result, task, "predictions")
    if prediction_path is None:
        return False
    metadata_by_id = doc_metadata_for_task(task)
    with prediction_path.open() as prediction_file:
        for prediction_line in prediction_file:
            return json.loads(prediction_line).get("native_id") in metadata_by_id
    return False


stale_jobs = []
matched_results = []
for result in raw_results:
    matched = [
        task
        for task in result.get("tasks", [])
        if not is_integer_code_task(task) or task_matches_source(result, task)
    ]
    dropped = len(result.get("tasks", [])) - len(matched)
    if dropped:
        stale_jobs.append(
            {
                "created_at": result["created_at"].strftime("%Y-%m-%d %H:%M"),
                "group": result.get("experiment_group"),
                "experiment_name": result.get("experiment_name"),
                "stale_tasks": dropped,
            }
        )
    if matched:
        matched_results.append({**result, "tasks": matched})
raw_results = matched_results
if stale_jobs:
    print("Dropped task results scored against an earlier dataset revision:")
    display(pd.DataFrame(stale_jobs))
if not raw_results:
    raise ValueError(
        "Every selected job was scored against a different dataset revision. "
        "Re-run the evals, or narrow RESULTS_SINCE to one generation of jobs."
    )

totals = defaultdict(
    lambda: {
        "score_sum": 0.0,
        "chance_sum": 0.0,
        "n": 0,
        "tasks": set(),
        "experiments": set(),
        "experiment_names": set(),
        "experiment_families": set(),
    }
)
missing_artifacts = []

for result in raw_results:
    training = training_metadata_for(result)

    for task in result.get("tasks", []):
        if not is_integer_code_task(task):
            continue
        prediction_path = artifact_file_for_task(result, task, "predictions")
        if prediction_path is None:
            missing_artifacts.append((result["experiment_id"], task["task"]))
            continue

        metric = task.get("primary_metric")
        metadata_by_id = doc_metadata_for_task(task)
        with prediction_path.open() as prediction_file:
            for prediction_line in prediction_file:
                prediction = json.loads(prediction_line)
                doc = metadata_by_id.get(prediction.get("native_id"))
                if doc is None:
                    source = task.get("config", {}).get("data_source") or {}
                    raise ValueError(
                        f"native_id {prediction.get('native_id')!r} from "
                        f"{result['experiment_id']} {task['task']} not found in "
                        f"{source.get('subset')}/{source.get('split')}"
                    )

                score = metric_value(prediction.get("instance_metrics", {}), metric)
                if score is None:
                    continue

                if not is_in_distribution_padding(doc):
                    continue
                evaluation = evaluation_metadata_for(doc)
                key = (
                    training["architecture"],
                    training["training_dataset"],
                    training["training_seed"],
                    evaluation["evaluation_dataset"],
                    evaluation["assignment_count"],
                )
                total = totals[key]
                total["score_sum"] += score
                total["chance_sum"] += chance_accuracy(prediction)
                total["n"] += 1
                total["tasks"].add(task["task"])
                total["experiments"].add(result["experiment_id"])
                experiment_name = result.get("experiment_name", result["experiment_id"])
                total["experiment_names"].add(experiment_name)
                total["experiment_families"].add(
                    experiment_family_name(experiment_name)
                )

if missing_artifacts:
    print(
        f"Skipped {len(missing_artifacts):,} task(s) without prediction artifacts."
    )

assignment_rows = []
for key, total in totals.items():
    (
        architecture,
        training_dataset,
        training_seed,
        evaluation_dataset,
        assignment_count,
    ) = key
    assignment_rows.append(
        {
            "architecture": architecture,
            "experiment_name": " | ".join(sorted(total["experiment_names"])),
            "experiment_family": " | ".join(sorted(total["experiment_families"])),
            "training_dataset": training_dataset,
            "training_seed": training_seed,
            "evaluation_dataset": evaluation_dataset,
            "assignment_count": assignment_count,
            "accuracy": total["score_sum"] / total["n"],
            "chance_total": total["chance_sum"],
            "examples": total["n"],
            "eval_tasks": len(total["tasks"]),
            "experiments": len(total["experiments"]),
        }
    )

assignment_df = pd.DataFrame(assignment_rows)
if assignment_df.empty:
    raise ValueError("No per-example results found for the selected jobs")

assignment_df["training_dataset"] = pd.Categorical(
    assignment_df["training_dataset"], categories=TRAIN_DATASET_ORDER, ordered=True
)
assignment_df["evaluation_dataset"] = pd.Categorical(
    assignment_df["evaluation_dataset"], categories=EVAL_DATASET_ORDER, ordered=True
)
assignment_df["series"] = assignment_df["architecture"]
assignment_df = assignment_df.sort_values(
    [
        "training_dataset",
        "training_seed",
        "evaluation_dataset",
        "architecture",
        "assignment_count",
    ]
).reset_index(drop=True)

print(
    f"Loaded {len(raw_results):,} experiments and aggregated {assignment_df['examples'].sum():,} examples."
)

seed_summary_df = (
    assignment_df.groupby(
        ["architecture", "training_dataset"],
        observed=True,
        as_index=False,
    )
    .agg(
        seed_count=("training_seed", "nunique"),
        seeds=("training_seed", lambda seeds: tuple(sorted(seeds.unique()))),
    )
    .sort_values(["training_dataset", "architecture"])
    .reset_index(drop=True)
)
display(seed_summary_df)

coverage_df = (
    assignment_df.groupby(
        [
            "experiment_name",
            "architecture",
            "training_dataset",
            "training_seed",
            "evaluation_dataset",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        assignment_values=("assignment_count", "nunique"),
        min_assignments=("assignment_count", "min"),
        max_assignments=("assignment_count", "max"),
        examples=("examples", "sum"),
        eval_tasks=("eval_tasks", "max"),
    )
    .sort_values(
        [
            "training_dataset",
            "training_seed",
            "evaluation_dataset",
            "architecture",
            "experiment_name",
        ]
    )
)
display(coverage_df)

## Accuracy by Assignment Count

Accuracy pools all examples in each five-assignment bin.

In [ ]:
assignment_min_power = int(np.floor(np.log2(ASSIGNMENT_AXIS_MIN)))
assignment_max_power = (
    int(np.floor(np.log2(assignment_df["assignment_count"].max()))) + 1
)
ASSIGNMENT_EDGES = 2.0 ** np.arange(assignment_min_power, assignment_max_power + 1)
assignment_df["assignment_bin"] = pd.cut(
    assignment_df["assignment_count"],
    ASSIGNMENT_EDGES,
    right=False,
    include_lowest=True,
)
assignment_df["assignment_mid"] = (
    assignment_df["assignment_bin"]
    .map(lambda value: np.sqrt(value.left * value.right) if pd.notna(value) else np.nan)
    .astype(float)
)
assignment_df["correct"] = assignment_df["accuracy"] * assignment_df["examples"]

# Pool across eval jobs: each context tier is its own experiment family, and
# keeping it as a key would leave one point per tier in the same bin.
plot_group_columns = [
    "architecture",
    "training_dataset",
    "evaluation_dataset",
    "series",
    "assignment_bin",
    "assignment_mid",
]
plot_df = assignment_df.groupby(plot_group_columns, observed=True, as_index=False).agg(
    correct=("correct", "sum"),
    chance_total=("chance_total", "sum"),
    examples=("examples", "sum"),
    seed_count=("training_seed", "nunique"),
)
plot_df = plot_df[plot_df["examples"] >= MIN_BIN_EXAMPLES].copy()
plot_df["accuracy"] = plot_df["correct"] / plot_df["examples"]
plot_df["ci_low"], plot_df["ci_high"] = proportion_confint(
    plot_df["correct"], plot_df["examples"], alpha=0.05, method="wilson"
)
plot_df["assignment_count"] = plot_df["assignment_mid"]


def configure_assignment_axis(ax: plt.Axes) -> None:
    ax.set_xscale("log", base=10)
    # Bins start at the dataset floor of 8, but the first populated bin's
    # geometric mid sits above 10, so open the axis at the decade tick.
    ax.set_xlim(10, ASSIGNMENT_EDGES[-1])
    add_assignment_reference_line(ax)
    ax.xaxis.set_major_locator(mtick.LogLocator(base=10, numticks=8))
    ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
    ax.xaxis.set_minor_locator(
        mtick.LogLocator(base=10, subs=(1, 2, 3, 4, 5, 6, 7, 8, 9), numticks=100)
    )
    ax.xaxis.set_minor_formatter(mtick.NullFormatter())


plot_df["training_dataset"] = plot_df["training_dataset"].map(DATASET_LABEL)
plot_df["evaluation_dataset"] = plot_df["evaluation_dataset"].map(DATASET_LABEL)
train_label_order = [DATASET_LABEL[name] for name in TRAIN_DATASET_ORDER]
eval_label_order = [DATASET_LABEL[name] for name in EVAL_DATASET_ORDER]
present_series = [series for series in SERIES_ORDER if series in set(plot_df["series"])]


def plot_series_curves(ax: plt.Axes, panel_df: pd.DataFrame, x_column: str) -> None:
    """Accuracy curves with Wilson bands; marker encodes the model variant."""
    for series in present_series:
        series_df = panel_df[panel_df["series"] == series].sort_values(x_column)
        if series_df.empty:
            continue
        ax.fill_between(
            series_df[x_column],
            series_df["ci_low"],
            series_df["ci_high"],
            color=SERIES_PALETTE[series],
            alpha=0.12,
            linewidth=0,
            zorder=1,
        )
        ax.plot(
            series_df[x_column],
            series_df["accuracy"],
            marker=SERIES_MARKERS[series],
            ms=3,
            lw=1.2,
            color=SERIES_PALETTE[series],
            zorder=2,
        )


def series_legend_handles() -> list[plt.Line2D]:
    return [
        plt.Line2D(
            [0],
            [0],
            color=SERIES_PALETTE[series],
            marker=SERIES_MARKERS[series],
            linewidth=1.5,
            label=SERIES_LABELS[series],
        )
        for series in present_series
    ]


fig = plt.figure(figsize=(9.6, 1.28 * len(TRAIN_DATASET_ORDER)))
grid = fig.add_gridspec(
    nrows=len(TRAIN_DATASET_ORDER),
    ncols=len(EVAL_DATASET_ORDER),
    right=0.72,
    hspace=0.35,
    wspace=0.1,
)

axes = []
for row_index, training_label in enumerate(train_label_order):
    row_axes = []
    for col_index, evaluation_label in enumerate(eval_label_order):
        share_axis = axes[0][col_index] if axes else None
        ax = fig.add_subplot(
            grid[row_index, col_index], sharex=share_axis, sharey=share_axis
        )
        row_axes.append(ax)

        panel_df = plot_df[
            (plot_df["training_dataset"] == training_label)
            & (plot_df["evaluation_dataset"] == evaluation_label)
        ]
        plot_series_curves(ax, panel_df, "assignment_count")

        ax.set_title(
            f"{training_label} \u2192 {evaluation_label}",
            fontsize=8,
            fontweight="regular",
        )
        add_chance_line(ax, panel_df)
        configure_assignment_axis(ax)
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.2)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.set_ylabel("Accuracy" if col_index == 0 else "")
        ax.tick_params(
            labelbottom=row_index == len(TRAIN_DATASET_ORDER) - 1,
            labelleft=col_index == 0,
        )

        if row_index == len(TRAIN_DATASET_ORDER) - 1 and col_index == 0:
            ax.set_xlabel("# assignments (log scale)")
        else:
            ax.set_xlabel(None)
    axes.append(row_axes)

legend_handles = series_legend_handles() + [
    plt.Line2D(
        [0],
        [0],
        color=ASSIGNMENT_REFERENCE_COLOR,
        linestyle=":",
        linewidth=1.45,
        label="training cutoff",
    ),
    plt.Line2D(
        [0],
        [0],
        color=CHANCE_LINE_COLOR,
        linestyle="--",
        linewidth=1.25,
        label="chance accuracy",
    ),
]
fig.legend(
    handles=legend_handles,
    title=None,
    loc="center left",
    bbox_to_anchor=(0.74, 0.5),
    frameon=False,
)

if SAVE_FIGURE:
    _ = aes.save_figure(FIGURE_PATH, fig=fig)

plt.show()

In [ ]:
# Matched-distribution view: retain only the train/eval diagonal in one row,
# rendered once for the three original conditions and once for the
# periodic-family contrast (plain periodic vs the sensitivity orderings).


def render_assignment_summary(train_datasets: list[str], figure_name: str) -> None:
    diagonal_fig = plt.figure(figsize=(2.56 * len(train_datasets) + 2.4, aes.FIG_HEIGHT_SINGLE_ROW_IN))
    diagonal_grid = diagonal_fig.add_gridspec(
        nrows=1,
        ncols=len(train_datasets),
        right=1 - 2.4 / (2.56 * len(train_datasets) + 2.4),
        wspace=0.1,
    )

    diagonal_axes = []
    for panel_index, training_dataset in enumerate(train_datasets):
        training_label = DATASET_LABEL[training_dataset]
        evaluation_label = DATASET_LABEL[matched_eval_dataset(training_dataset)]
        share_y_axis = diagonal_axes[0] if diagonal_axes else None
        ax = diagonal_fig.add_subplot(diagonal_grid[0, panel_index], sharey=share_y_axis)
        diagonal_axes.append(ax)

        panel_df = plot_df[
            (plot_df["training_dataset"] == training_label)
            & (plot_df["evaluation_dataset"] == evaluation_label)
        ]
        plot_series_curves(ax, panel_df, "assignment_count")

        ax.set_title(
            f"{training_label} \u2192 {evaluation_label}",
            fontsize=8,
            fontweight="regular",
        )
        add_chance_line(ax, panel_df)
        configure_assignment_axis(ax)
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.2)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.set_ylabel("Accuracy" if panel_index == 0 else "")
        ax.tick_params(labelleft=panel_index == 0)

        if panel_index == 0:
            ax.set_xlabel("# assignments (log scale)")
        else:
            ax.set_xlabel(None)

    diagonal_legend_handles = series_legend_handles() + [
        plt.Line2D(
            [0],
            [0],
            color=ASSIGNMENT_REFERENCE_COLOR,
            linestyle=":",
            linewidth=1.45,
            label="training cutoff",
        ),
        plt.Line2D(
            [0],
            [0],
            color=CHANCE_LINE_COLOR,
            linestyle="--",
            linewidth=1.25,
            label="chance accuracy",
        ),
    ]
    diagonal_fig.legend(
        handles=diagonal_legend_handles,
        title=None,
        loc="center left",
        bbox_to_anchor=(1 - 2.2 / (2.56 * len(train_datasets) + 2.4), 0.5),
        frameon=False,
    )

    diagonal_fig.suptitle(
        "Directly-trained models on StateBench",
        fontweight="bold",
        ha="left",
        fontsize=10,
        x=0.125,
        y=1.25
    )

    if SAVE_FIGURE:
        _ = aes.save_figure(
            PROJECT_ROOT / "state-bench-paper" / "figures" / figure_name, fig=diagonal_fig
        )

    plt.show()


render_assignment_summary(ORIGINAL_TRAIN_DATASETS, "dt_accuracy_diagonal_by_assignments")
render_assignment_summary(PERIODIC_TRAIN_DATASETS, "dt_accuracy_periodic_by_assignments")


## Accuracy by Context Length

These views mirror the assignment-count panels, binning by the evaluation token strata (as in pt_results, so every bin carries the same sampled count per task) and using the long-context notebook's base-10 log axis styling.

In [ ]:
# Evaluation tiers, ascending; these double as the context-length bins below,
# matching pt_results: each tier is sampled to the same number of instances
# per task, so tier bins carry equal counts by construction.
TIER_ORDER = [
    "tokens_under_2k",
    "tokens_4k",
    "tokens_8k",
    "tokens_16k",
    "tokens_32k",
    "tokens_64k",
    "tokens_128k",
    "tokens_256k",
    "tokens_512k",
    "tokens_1m",
]
TIER_RE = re.compile(r"_(?P<tier>tokens_.+)$")

context_rows = []
for result in raw_results:
    training = training_metadata_for(result)
    for task in result.get("tasks", []):
        if not is_integer_code_task(task):
            continue
        prediction_path = artifact_file_for_task(result, task, "predictions")
        if prediction_path is None:
            continue
        tier_match = TIER_RE.search(str(task.get("task", "")))
        if tier_match is None:
            raise ValueError(f"No context tier in task name: {task.get('task')!r}")
        tier = tier_match.group("tier")
        metadata_by_id = doc_metadata_for_task(task)
        with prediction_path.open() as predictions:
            for prediction_line in predictions:
                prediction = json.loads(prediction_line)
                score = metric_value(
                    prediction.get("instance_metrics", {}), task.get("primary_metric")
                )
                doc = metadata_by_id.get(prediction.get("native_id"))
                if score is None or doc is None or not is_in_distribution_padding(doc):
                    continue
                evaluation = evaluation_metadata_for(doc)
                context_rows.append(
                    {
                        "chance": chance_accuracy(prediction),
                        "experiment_family": experiment_family_name(
                            result.get("experiment_name", result["experiment_id"])
                        ),
                        "architecture": training["architecture"],
                        "training_dataset": training["training_dataset"],
                        "evaluation_dataset": evaluation["evaluation_dataset"],
                        "series": training["architecture"],
                        "score": score,
                        "tier": tier,
                        "context_length": int(doc["token_length"]),
                    }
                )

context_df = pd.DataFrame(context_rows)
# Context bins are the evaluation tiers rather than free power-of-two bins,
# matching pt_results: each tier is an equal-count sample per task, and each
# bin plots at its tier's median observed context length.
present_tiers = [t for t in TIER_ORDER if t in set(context_df["tier"])]
context_df["context_bin"] = pd.Categorical(context_df["tier"], present_tiers, ordered=True)
tier_median_context = context_df.groupby("context_bin", observed=True)["context_length"].median()
context_df["context_mid"] = context_df["context_bin"].map(tier_median_context).astype(float)
context_plot_df = (
    context_df.groupby(
        [
            "architecture",
            "training_dataset",
            "evaluation_dataset",
            "series",
            "context_bin",
            "context_mid",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        correct=("score", "sum"),
        chance_total=("chance", "sum"),
        examples=("score", "size"),
    )
    .query("examples >= @MIN_BIN_EXAMPLES")
)
context_plot_df["accuracy"] = context_plot_df["correct"] / context_plot_df["examples"]
context_plot_df["ci_low"], context_plot_df["ci_high"] = proportion_confint(
    context_plot_df["correct"], context_plot_df["examples"], alpha=0.05, method="wilson"
)
context_plot_df["training_dataset"] = context_plot_df["training_dataset"].map(
    DATASET_LABEL
)
context_plot_df["evaluation_dataset"] = context_plot_df["evaluation_dataset"].map(
    DATASET_LABEL
)


def configure_context_axis(ax: plt.Axes) -> None:
    ax.set_xscale("log", base=10)
    # Limits follow the plotted tier medians with a small log-scale margin.
    ax.set_xlim(tier_median_context.min() / 1.4, tier_median_context.max() * 1.4)
    ax.xaxis.set_major_locator(mtick.LogLocator(base=10, numticks=8))
    ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
    ax.xaxis.set_minor_locator(
        mtick.LogLocator(base=10, subs=(1, 2, 3, 4, 5, 6, 7, 8, 9), numticks=100)
    )
    ax.xaxis.set_minor_formatter(mtick.NullFormatter())


def plot_context_curves(ax: plt.Axes, panel_df: pd.DataFrame) -> None:
    for series in present_series:
        curve = panel_df[panel_df["series"] == series].sort_values("context_mid")
        x = curve["context_mid"].to_numpy(float)
        ax.fill_between(
            x,
            curve["ci_low"].to_numpy(float),
            curve["ci_high"].to_numpy(float),
            color=SERIES_PALETTE[series],
            alpha=0.12,
            zorder=1,
        )
        ax.plot(
            x,
            curve["accuracy"].to_numpy(float),
            marker=SERIES_MARKERS[series],
            ms=2.5,
            lw=1.2,
            color=SERIES_PALETTE[series],
            zorder=2,
        )


context_legend_handles = series_legend_handles() + [
    plt.Line2D(
        [0],
        [0],
        color=CHANCE_LINE_COLOR,
        linestyle="--",
        linewidth=1.25,
        label="chance accuracy",
    )
]
fig = plt.figure(figsize=(9.6, 1.28 * len(TRAIN_DATASET_ORDER)))
grid = fig.add_gridspec(
    nrows=len(TRAIN_DATASET_ORDER),
    ncols=len(EVAL_DATASET_ORDER),
    right=0.72,
    hspace=0.35,
    wspace=0.1,
)
axes = []
for row_index, training_label in enumerate(train_label_order):
    row_axes = []
    for col_index, evaluation_label in enumerate(eval_label_order):
        ax = fig.add_subplot(
            grid[row_index, col_index],
            sharex=axes[0][col_index] if axes else None,
            sharey=axes[0][col_index] if axes else None,
        )
        row_axes.append(ax)
        panel_df = context_plot_df[
            (context_plot_df["training_dataset"] == training_label)
            & (context_plot_df["evaluation_dataset"] == evaluation_label)
        ]
        plot_context_curves(ax, panel_df)
        add_chance_line(ax, panel_df)
        configure_context_axis(ax)
        ax.set_title(
            f"{training_label} \u2192 {evaluation_label}",
            fontsize=8,
            fontweight="regular",
        )
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.2)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.set_ylabel("Accuracy" if col_index == 0 else "")
        ax.tick_params(
            labelbottom=row_index == len(TRAIN_DATASET_ORDER) - 1,
            labelleft=col_index == 0,
        )
        ax.set_xlabel(
            "Context length (log scale)"
            if row_index == len(TRAIN_DATASET_ORDER) - 1 and col_index == 0
            else None
        )
    axes.append(row_axes)
fig.legend(
    handles=context_legend_handles,
    loc="center left",
    bbox_to_anchor=(0.74, 0.5),
    frameon=False,
)
_ = aes.save_figure(
    PROJECT_ROOT / "state-bench-paper" / "figures" / "dt_accuracy_by_context_length", fig=fig
)
plt.show()

def render_context_summary(train_datasets: list[str], figure_name: str) -> None:
    diagonal_fig = plt.figure(figsize=(2.56 * len(train_datasets) + 2.4, aes.FIG_HEIGHT_SINGLE_ROW_IN))
    diagonal_grid = diagonal_fig.add_gridspec(
        nrows=1,
        ncols=len(train_datasets),
        right=1 - 2.4 / (2.56 * len(train_datasets) + 2.4),
        wspace=0.1,
    )
    diagonal_axes = []
    for panel_index, training_dataset in enumerate(train_datasets):
        training_label = DATASET_LABEL[training_dataset]
        evaluation_label = DATASET_LABEL[matched_eval_dataset(training_dataset)]
        ax = diagonal_fig.add_subplot(
            diagonal_grid[0, panel_index],
            sharey=diagonal_axes[0] if diagonal_axes else None,
        )
        diagonal_axes.append(ax)
        panel_df = context_plot_df[
            (context_plot_df["training_dataset"] == training_label)
            & (context_plot_df["evaluation_dataset"] == evaluation_label)
        ]
        plot_context_curves(ax, panel_df)
        add_chance_line(ax, panel_df)
        configure_context_axis(ax)
        ax.set_title(
            f"{training_label} \u2192 {evaluation_label}",
            fontsize=8,
            fontweight="regular",
        )
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.2)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.set_ylabel("Accuracy" if panel_index == 0 else "")
        ax.tick_params(labelleft=panel_index == 0)
        ax.set_xlabel("Context length (log scale)" if panel_index == 0 else None)
    diagonal_fig.legend(
        handles=context_legend_handles,
        loc="center left",
        bbox_to_anchor=(1 - 2.2 / (2.56 * len(train_datasets) + 2.4), 0.5),
        frameon=False,
    )
    _ = aes.save_figure(
        PROJECT_ROOT / "state-bench-paper" / "figures" / figure_name, fig=diagonal_fig
    )
    plt.show()


render_context_summary(ORIGINAL_TRAIN_DATASETS, "dt_accuracy_diagonal_by_context_length")
render_context_summary(PERIODIC_TRAIN_DATASETS, "dt_accuracy_periodic_by_context_length")


## Periodic Accuracy by Dependency Length

This joins periodic evaluation samples to their source dataset to recover maximum dependency length, using the long-context notebook's bins and plot styling.

In [ ]:
DEPENDENCY_EDGES = np.r_[-0.5, 0.5, 1.5, 2.5, 2 ** np.arange(2, 16) + 0.5]

dependency_rows = []
for result in raw_results:
    training = training_metadata_for(result)
    for task in result.get("tasks", []):
        source = task.get("config", {}).get("data_source") or {}
        if (
            not is_integer_code_task(task)
            or source.get("subset") != "integer-code--periodic"
        ):
            continue
        prediction_path = artifact_file_for_task(result, task, "predictions")
        if prediction_path is None:
            continue
        metadata_by_id = doc_metadata_for_task(task)
        with prediction_path.open() as predictions:
            for prediction_line in predictions:
                prediction = json.loads(prediction_line)
                score = metric_value(
                    prediction.get("instance_metrics", {}), task.get("primary_metric")
                )
                doc = metadata_by_id.get(prediction.get("native_id"))
                if (
                    score is not None
                    and doc is not None
                    and is_in_distribution_padding(doc)
                ):
                    dependency_rows.append(
                        {
                            "chance": chance_accuracy(prediction),
                            "training_dataset": training["training_dataset"],
                            "series": training["architecture"],
                            "accuracy": score,
                            "dependency_length": doc["max_dependency_length"],
                        }
                    )

dependency_df = pd.DataFrame(dependency_rows)
dependency_df["dependency_bin"] = pd.cut(
    dependency_df["dependency_length"],
    DEPENDENCY_EDGES,
    right=False,
    include_lowest=True,
)
periodic_dependency_df = (
    dependency_df.groupby(
        ["training_dataset", "series", "dependency_bin"], observed=True, as_index=False
    )
    .agg(
        accuracy=("accuracy", "mean"),
        successes=("accuracy", "sum"),
        chance_total=("chance", "sum"),
        examples=("accuracy", "size"),
        dependency_mid=("dependency_length", "median"),
    )
    .query("examples >= @MIN_BIN_EXAMPLES")
)
periodic_dependency_df["ci_low"], periodic_dependency_df["ci_high"] = (
    proportion_confint(
        periodic_dependency_df["successes"],
        periodic_dependency_df["examples"],
        alpha=0.05,
        method="wilson",
    )
)


In [ ]:
def render_dependency_summary(train_datasets: list[str], figure_name: str) -> None:
    fig = plt.figure(figsize=(2.56 * len(train_datasets) + 2.4, aes.FIG_HEIGHT_SINGLE_ROW_IN))
    grid = fig.add_gridspec(
        nrows=1,
        ncols=len(train_datasets),
        right=1 - 2.4 / (2.56 * len(train_datasets) + 2.4),
        wspace=0.1,
    )
    axes = []
    for panel_index, training_dataset in enumerate(train_datasets):
        ax = fig.add_subplot(grid[0, panel_index], sharey=axes[0] if axes else None)
        axes.append(ax)
        subset = periodic_dependency_df[
            periodic_dependency_df["training_dataset"] == training_dataset
        ]
        for series in present_series:
            curve = subset[subset["series"] == series].sort_values("dependency_mid")
            x = curve["dependency_mid"].to_numpy(float)
            ax.fill_between(
                x,
                curve["ci_low"].to_numpy(float),
                curve["ci_high"].to_numpy(float),
                color=SERIES_PALETTE[series],
                alpha=0.12,
                zorder=1,
            )
            ax.plot(
                x,
                curve["accuracy"].to_numpy(float),
                marker=SERIES_MARKERS[series],
                ms=3,
                lw=1.5,
                color=SERIES_PALETTE[series],
                label=SERIES_LABELS[series],
                zorder=2,
            )
        add_chance_line(ax, subset)
        ax.set_xscale("symlog", base=10, linthresh=1)
        ax.set_xlim(1, None)
        ax.set_xticks(
            (np.arange(2, 10)[None, :] * 10 ** np.arange(4)[:, None]).ravel(), minor=True
        )
        ax.tick_params(axis="x", which="minor", length=3)
        ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
        ax.set_title(
            f"{DATASET_LABEL[training_dataset]} \u2192 Periodic",
            fontsize=8,
            fontweight="regular",
        )
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.grid(alpha=0.2)
        ax.set_ylabel("Accuracy" if panel_index == 0 else "")
        ax.tick_params(labelleft=panel_index == 0)
        ax.set_xlabel("Maximum dependency length" if panel_index == 0 else None)
    fig.legend(
        handles=series_legend_handles()
        + [
            plt.Line2D(
                [0],
                [0],
                color=CHANCE_LINE_COLOR,
                linestyle="--",
                linewidth=1.25,
                label="chance accuracy",
            )
        ],
        loc="center left",
        frameon=False,
        bbox_to_anchor=(1 - 2.2 / (2.56 * len(train_datasets) + 2.4), 0.5),
    )
    fig.suptitle(
        "Periodic eval accuracy by maximum dependency length",
        fontweight="bold",
        ha="left",
        x=0.125,
        y=1.3,
    )
    sns.despine()
    _ = aes.save_figure(
        PROJECT_ROOT / "state-bench-paper" / "figures" / figure_name,
        fig=fig,
    )
    plt.show()


render_dependency_summary(ORIGINAL_TRAIN_DATASETS, "dt_accuracy_by_dependency_length")
render_dependency_summary(PERIODIC_TRAIN_DATASETS, "dt_accuracy_periodic_by_dependency_length")


## Periodic Accuracy by Sample Sensitivity

Models trained on the three periodic corpora (plain, sensitivity-shuffled, sensitivity-curriculum), evaluated on periodic examples, binned by each example's sensitivity coverage — the fraction of live program state that influences the queried outputs. The shuffled and curriculum corpora hold the same tokens and differ only in presentation order, so gaps between those two panels are ordering effects.


In [ ]:
PERIODIC_TRAINING_ORDER = list(PERIODIC_TRAIN_DATASETS)
SENSITIVITY_EDGES = np.linspace(0.0, 1.0, 11)

sensitivity_rows = []
for result in raw_results:
    training = training_metadata_for(result)
    if training["training_dataset"] not in PERIODIC_TRAINING_ORDER:
        continue
    for task in result.get("tasks", []):
        source = task.get("config", {}).get("data_source") or {}
        if (
            not is_integer_code_task(task)
            or source.get("subset") != "integer-code--periodic"
        ):
            continue
        prediction_path = artifact_file_for_task(result, task, "predictions")
        if prediction_path is None:
            continue
        metadata_by_id = doc_metadata_for_task(task)
        with prediction_path.open() as predictions:
            for prediction_line in predictions:
                prediction = json.loads(prediction_line)
                score = metric_value(
                    prediction.get("instance_metrics", {}), task.get("primary_metric")
                )
                doc = metadata_by_id.get(prediction.get("native_id"))
                if (
                    score is not None
                    and doc is not None
                    and is_in_distribution_padding(doc)
                ):
                    sensitivity_rows.append(
                        {
                            "chance": chance_accuracy(prediction),
                            "training_dataset": training["training_dataset"],
                            "series": training["architecture"],
                            "accuracy": score,
                            "sensitivity_coverage": float(doc["sensitivity_coverage"]),
                        }
                    )

sensitivity_df = pd.DataFrame(sensitivity_rows)
sensitivity_df["sensitivity_bin"] = pd.cut(
    sensitivity_df["sensitivity_coverage"],
    SENSITIVITY_EDGES,
    right=True,
    include_lowest=True,
)
periodic_sensitivity_df = (
    sensitivity_df.groupby(
        ["training_dataset", "series", "sensitivity_bin"], observed=True, as_index=False
    )
    .agg(
        accuracy=("accuracy", "mean"),
        successes=("accuracy", "sum"),
        chance_total=("chance", "sum"),
        examples=("accuracy", "size"),
        sensitivity_mid=("sensitivity_coverage", "median"),
    )
    .query("examples >= @MIN_BIN_EXAMPLES")
)
periodic_sensitivity_df["ci_low"], periodic_sensitivity_df["ci_high"] = (
    proportion_confint(
        periodic_sensitivity_df["successes"],
        periodic_sensitivity_df["examples"],
        alpha=0.05,
        method="wilson",
    )
)


In [ ]:
fig = plt.figure(figsize=(3.2 * len(PERIODIC_TRAINING_ORDER), aes.FIG_HEIGHT_SINGLE_ROW_IN))
grid = fig.add_gridspec(
    nrows=1, ncols=len(PERIODIC_TRAINING_ORDER), right=0.72, wspace=0.1
)
axes = []
for panel_index, training_dataset in enumerate(PERIODIC_TRAINING_ORDER):
    ax = fig.add_subplot(grid[0, panel_index], sharey=axes[0] if axes else None)
    axes.append(ax)
    subset = periodic_sensitivity_df[
        periodic_sensitivity_df["training_dataset"] == training_dataset
    ]
    for series in present_series:
        curve = subset[subset["series"] == series].sort_values("sensitivity_mid")
        if curve.empty:
            continue
        x = curve["sensitivity_mid"].to_numpy(float)
        ax.fill_between(
            x,
            curve["ci_low"].to_numpy(float),
            curve["ci_high"].to_numpy(float),
            color=SERIES_PALETTE[series],
            alpha=0.12,
            zorder=1,
        )
        ax.plot(
            x,
            curve["accuracy"].to_numpy(float),
            marker=SERIES_MARKERS[series],
            ms=3,
            lw=1.5,
            color=SERIES_PALETTE[series],
            zorder=2,
        )
    add_chance_line(ax, subset)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1.02)
    ax.set_xticks([0.0, 0.5, 1.0])
    ax.set_xticks([0.25, 0.75], minor=True)
    ax.xaxis.set_major_formatter(aes.PCT_FORMATTER)
    if panel_index:
        # The panels sit flush; drop the leading 0% so it doesn't collide with
        # the previous panel's 100%.
        ax.set_xticklabels(["", "50%", "100%"])
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.set_title(
        f"{DATASET_LABEL[training_dataset]} \u2192 Periodic",
        fontsize=8,
        fontweight="regular",
    )
    ax.grid(alpha=0.2)
    ax.set_ylabel("Accuracy" if panel_index == 0 else "")
    ax.tick_params(labelleft=panel_index == 0)
    ax.set_xlabel("Sensitivity coverage" if panel_index == 0 else None)
fig.legend(
    handles=series_legend_handles()
    + [
        plt.Line2D(
            [0],
            [0],
            color=CHANCE_LINE_COLOR,
            linestyle="--",
            linewidth=1.25,
            label="chance accuracy",
        )
    ],
    loc="center left",
    frameon=False,
    bbox_to_anchor=(0.74, 0.5),
)
fig.suptitle(
    "Periodic eval accuracy by sample sensitivity",
    fontweight="bold",
    ha="left",
    x=0.125,
    y=1.3,
)
sns.despine()
_ = aes.save_figure(
    PROJECT_ROOT / "state-bench-paper" / "figures" / "dt_accuracy_by_sensitivity",
    fig=fig,
)
plt.show()


## Accuracy Regression

This binomial-logit regression uses one row per Integer Code evaluation sample/model pair. It predicts binary accuracy from architecture, maximum dependency length, context length, assignment count, evaluation monoid complexity, and padding amount. Continuous predictors are log-transformed with `log1p` and standardized, so their odds ratios correspond to a one-standard-deviation increase on that scale. Standard errors are clustered by evaluation-sample identity.

In [ ]:
MODEL_PATH_METADATA_RE = re.compile(
    r"/(?P<parameter_scale_millions>\d+)M/"
    r"(?P<architecture>transformer-rope|transformer-nope|hybrid|gdn-sdp|gdn-full)/"
    r"integer-code--(?P<training_dataset>r-trivial|aperiodic"
    r"|periodic--sens-shuffled|periodic--sens-curriculum|periodic)/"
    r"Cx(?P<config_cx>\d+)/init_seed(?P<training_seed>\d+)/"
    r"step(?P<checkpoint_step>\d+)-hf(?:/|$)",
    flags=re.IGNORECASE,
)


def parsed_model_metadata(result: dict) -> dict:
    """Parse the scalar configuration fields encoded in the stored model path."""
    model_path = str(result.get("model", ""))
    match = MODEL_PATH_METADATA_RE.search(model_path)
    if match is None:
        raise ValueError(f"Could not parse model metadata from: {model_path}")

    metadata = match.groupdict()
    metadata["architecture"] = metadata["architecture"].lower()
    metadata["training_dataset"] = (
        metadata["training_dataset"].lower().replace("--sens-", "-sens-")
    )
    for name in set(metadata) - {"architecture", "training_dataset"}:
        metadata[name] = int(metadata[name])
    return metadata


regression_rows = []
for result in raw_results:
    model_metadata = parsed_model_metadata(result)

    for task in result.get("tasks", []):
        if not is_integer_code_task(task):
            continue
        prediction_path = artifact_file_for_task(result, task, "predictions")
        if prediction_path is None:
            continue

        metric = task.get("primary_metric")
        metadata_by_id = doc_metadata_for_task(task)
        with prediction_path.open() as prediction_file:
            for prediction_line in prediction_file:
                prediction = json.loads(prediction_line)
                doc = metadata_by_id.get(prediction.get("native_id"))
                if doc is None or not is_in_distribution_padding(doc):
                    continue

                score = metric_value(prediction.get("instance_metrics", {}), metric)
                if score is None:
                    continue
                if not math.isclose(score, 0.0) and not math.isclose(score, 1.0):
                    raise ValueError(
                        f"Binomial regression requires 0/1 accuracy; got {score}"
                    )

                evaluation = evaluation_metadata_for(doc)
                sample_native_id = prediction.get("native_id", prediction.get("doc_id"))
                regression_rows.append(
                    {
                        "accuracy": int(round(score)),
                        "experiment_id": result["experiment_id"],
                        "sample_key": f"{task['task']}::{sample_native_id}",
                        **model_metadata,
                        **evaluation,
                        "text_length": doc["token_length"],
                        "padding_amount": doc["num_filler_lines"],
                    }
                )

regression_df = pd.DataFrame(regression_rows)
if regression_df.empty:
    raise ValueError("No rows are available for regression.")

print(
    f"Regression data: {len(regression_df):,} sample/model observations, "
    f"{regression_df['sample_key'].nunique():,} unique evaluation samples, and "
    f"{regression_df['experiment_id'].nunique():,} model runs."
)

In [ ]:
model_rows = []
for result in raw_results:
    architecture = training_metadata_for(result)["architecture"]
    for task in result.get("tasks", []):
        if not is_integer_code_task(task):
            continue
        prediction_path = artifact_file_for_task(result, task, "predictions")
        if prediction_path is None:
            continue
        metadata_by_id = doc_metadata_for_task(task)
        with prediction_path.open() as predictions:
            for prediction_line in predictions:
                prediction = json.loads(prediction_line)
                score = metric_value(prediction.get("instance_metrics", {}), task.get("primary_metric"))
                doc = metadata_by_id.get(prediction.get("native_id"))
                if score is None or doc is None or not is_in_distribution_padding(doc):
                    continue
                model_rows.append({"accuracy": int(round(score)), "sample_key": f"{task['task']}::{prediction.get('native_id')}", "architecture": architecture, "max_dependency_length": doc["max_dependency_length"], "context_length": doc["token_length"], "assignment_count": doc["num_assignments"], "monoid_complexity": doc["complexity"], "padding_amount": doc["num_filler_lines"]})

model_df = pd.DataFrame(model_rows).dropna().copy()
for predictor in ["max_dependency_length", "context_length", "assignment_count", "padding_amount"]:
    transformed = np.log1p(model_df[predictor].astype(float))
    model_df[f"z_{predictor}"] = (transformed - transformed.mean()) / transformed.std()

formula = (
    'accuracy ~ C(architecture, Treatment(reference="transformer-rope"))'
    ' + z_max_dependency_length + z_context_length + z_assignment_count'
    ' + C(monoid_complexity, Treatment(reference="r-trivial")) + z_padding_amount'
)
accuracy_fit = smf.glm(formula=formula, data=model_df, family=sm.families.Binomial()).fit(cov_type="cluster", cov_kwds={"groups": model_df["sample_key"]})
regression_table = pd.DataFrame({"term": accuracy_fit.params.index, "odds_ratio": np.exp(accuracy_fit.params.to_numpy()), "ci_low": np.exp(accuracy_fit.conf_int().iloc[:, 0].to_numpy()), "ci_high": np.exp(accuracy_fit.conf_int().iloc[:, 1].to_numpy()), "p_value": accuracy_fit.pvalues.to_numpy()})
regression_table = regression_table[regression_table["term"] != "Intercept"]
display(regression_table.style.format({"odds_ratio": "{:.3f}", "ci_low": "{:.3f}", "ci_high": "{:.3f}", "p_value": "{:.3g}"}))
display(pd.DataFrame({"observations": [int(accuracy_fit.nobs)], "unique_samples": [model_df["sample_key"].nunique()], "aic": [accuracy_fit.aic]}).style.format({"aic": "{:.1f}"}))


In [ ]:
"""
PREDICTOR_SPECS = {
    # Sample metadata
    "assignment_count": ("numeric", "sample", "screen + joint"),
    "text_length": ("numeric", "sample", "screen + joint"),
    "gold_idx": ("categorical", "sample", "screen + joint"),
    "evaluation_dataset": ("categorical", "sample", "screen + joint"),
    "query_length": ("numeric", "sample", "screen only"),
    "prefix_length": ("numeric", "sample", "alias of query_length"),
    "gold_text_length": ("numeric", "sample", "screen + joint"),
    "choice_count": ("numeric", "sample", "screen if varying"),
    "alternative_count": ("numeric", "sample", "screen if varying"),
    "distractor_count": ("numeric", "sample", "screen if varying"),
    "sample_dataset": ("categorical", "sample", "screen if varying"),
    "eval_name": ("categorical", "sample", "alias of evaluation fields"),
    "file_prefix": ("categorical", "sample", "alias of eval_name"),
    # Model metadata
    "architecture": ("categorical", "model", "screen + joint"),
    "training_dataset": ("categorical", "model", "screen + joint"),
    "training_seed": ("categorical", "model", "screen + joint"),
    "parameter_scale_millions": ("numeric", "model", "screen if varying"),
    "config_cx": ("numeric", "model", "screen if varying"),
    "checkpoint_step": ("numeric", "model", "screen only; aliased in joint model"),
}

ALIASED_SCREEN_PREDICTORS = {
    "prefix_length",
    "gold_text_length",
    "eval_name",
    "file_prefix",
}

inventory_rows = []
for predictor, (kind, family, planned_handling) in PREDICTOR_SPECS.items():
    unique_values = regression_df[predictor].nunique(dropna=True)
    if unique_values < 2:
        regression_status = "not estimable (constant)"
    elif predictor in ALIASED_SCREEN_PREDICTORS:
        regression_status = planned_handling
    else:
        regression_status = planned_handling
    inventory_rows.append(
        {
            "family": family,
            "metadata": predictor,
            "type": kind,
            "unique_values": unique_values,
            "missing": int(regression_df[predictor].isna().sum()),
            "regression_handling": regression_status,
        }
    )

metadata_inventory_df = pd.DataFrame(inventory_rows).sort_values(["family", "metadata"])
display(metadata_inventory_df)
print(
    "Also excluded by design: raw query/prefix/gold/choice text and "
    "request, document, sample, experiment, and model-path identifiers."
)
"""

In [ ]:
"""
    confidence_interval = fit.conf_int()
    tidy = pd.DataFrame(
        {
            "analysis": analysis,
            "metadata": metadata,
            "term": fit.params.index,
            "log_odds": fit.params.to_numpy(),
            "std_error": fit.bse.to_numpy(),
            "p_value": fit.pvalues.to_numpy(),
            "odds_ratio": np.exp(fit.params.to_numpy()),
            "ci_low": np.exp(confidence_interval.iloc[:, 0].to_numpy()),
            "ci_high": np.exp(confidence_interval.iloc[:, 1].to_numpy()),
        }
    )
    return tidy[tidy["term"] != "Intercept"].reset_index(drop=True)


screening_tables = []
screening_model_summaries = []
for predictor, (kind, family, _) in PREDICTOR_SPECS.items():
    if predictor in ALIASED_SCREEN_PREDICTORS:
        continue

    model_data = regression_df[["accuracy", "sample_key", predictor]].dropna().copy()
    if model_data[predictor].nunique() < 2:
        continue

    if kind == "numeric":
        predictor_sd = model_data[predictor].std()
        if not np.isfinite(predictor_sd) or predictor_sd == 0:
            continue
        formula_predictor = f"z_{predictor}"
        model_data[formula_predictor] = (
            model_data[predictor] - model_data[predictor].mean()
        ) / predictor_sd
    else:
        formula_predictor = f"C({predictor})"

    fit = smf.glm(
        formula=f"accuracy ~ {formula_predictor}",
        data=model_data,
        family=sm.families.Binomial(),
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": model_data["sample_key"]},
    )
    screening_tables.append(tidy_binomial_fit(fit, "screening", predictor))
    screening_model_summaries.append(
        {
            "family": family,
            "metadata": predictor,
            "observations": int(fit.nobs),
            "terms": len(fit.params) - 1,
            "aic": fit.aic,
        }
    )

screening_regression_df = pd.concat(screening_tables, ignore_index=True)
screening_regression_df["fdr_q_value"] = multipletests(
    screening_regression_df["p_value"], method="fdr_bh"
)[1]
screening_regression_df["significant_at_fdr_0.05"] = (
    screening_regression_df["fdr_q_value"] < 0.05
)

display(
    screening_regression_df.sort_values(["fdr_q_value", "metadata", "term"])[
        [
            "metadata",
            "term",
            "odds_ratio",
            "ci_low",
            "ci_high",
            "p_value",
            "fdr_q_value",
            "significant_at_fdr_0.05",
        ]
    ].style.format(
        {
            "odds_ratio": "{:.3f}",
            "ci_low": "{:.3f}",
            "ci_high": "{:.3f}",
            "p_value": "{:.3g}",
            "fdr_q_value": "{:.3g}",
        }
    )
)
"""

In [ ]:
"""
JOINT_NUMERIC_PREDICTORS = [
    "assignment_count",
    "text_length",
    "gold_text_length",
]
for predictor in JOINT_NUMERIC_PREDICTORS:
    joint_df[f"z_{predictor}"] = (
        joint_df[predictor] - joint_df[predictor].mean()
    ) / joint_df[predictor].std()

joint_terms = [
    'C(architecture, Treatment(reference="transformer"))',
    (
        'C(training_dataset, Treatment(reference="r-trivial"))'
        ' * C(evaluation_dataset, Treatment(reference="r-trivial"))'
    ),
    "z_assignment_count",
    "z_text_length",
    "z_gold_text_length",
    "C(gold_idx, Treatment(reference=0))",
]
joint_formula = "accuracy ~ " + " + ".join(joint_terms)
joint_fit = smf.glm(
    formula=joint_formula,
    data=joint_df,
    family=sm.families.Binomial(),
).fit(
    cov_type="cluster",
    cov_kwds={"groups": joint_df["sample_key"]},
)

joint_regression_df = tidy_binomial_fit(joint_fit, "joint", "all")
joint_regression_df["significant_at_0.05"] = joint_regression_df["p_value"] < 0.05
display(
    joint_regression_df[
        [
            "term",
            "odds_ratio",
            "ci_low",
            "ci_high",
            "p_value",
            "significant_at_0.05",
        ]
    ].style.format(
        {
            "odds_ratio": "{:.3f}",
            "ci_low": "{:.3f}",
            "ci_high": "{:.3f}",
            "p_value": "{:.3g}",
        }
    )
)

joint_diagnostics_df = pd.DataFrame(
    {
        "observations": [int(joint_fit.nobs)],
        "unique_samples": [joint_df["sample_key"].nunique()],
        "model_runs": [joint_df["experiment_id"].nunique()],
        "parameters": [len(joint_fit.params)],
        "aic": [joint_fit.aic],
        "deviance": [joint_fit.deviance],
        "null_deviance": [joint_fit.null_deviance],
    }
)
display(
    joint_diagnostics_df.style.format(
        {"aic": "{:.1f}", "deviance": "{:.1f}", "null_deviance": "{:.1f}"}
    )
)
"""